# NL2SH — Qwen2.5-Coder-3B-Instruct Supervised Fine-Tuning (SFT)

**Project:** NL2SH (Natural Language to Shell)  
**Base Model:** `Qwen/Qwen2.5-Coder-3B-Instruct`  
**Method:** Parameter-Efficient Fine-Tuning (PEFT) with LoRA / QLoRA (4-bit BitsAndBytes)  
**Dataset:** NL2SH Clean SFT Split (28,167 train, 3,129 validation records)  
**Target Hardware:** Google Colab Free / Pro (NVIDIA T4 16GB, L4 24GB, or A100 40GB)

---

### SFT Objectives & Constraints
- **Mapping:** Natural-language instruction -> exact executable Bash command (zero conversational filler).
- **Loss Masking:** Response-only loss masking ensures gradients are computed strictly on generated Bash commands.
- **Sequence Length:** 512 tokens accommodates 100.0% of training and validation records with zero target truncation.
- **Persistence:** Checkpoints and final adapters are saved directly to Google Drive with automated resumption support.
- **Holdout Integrity:** The official 300-record benchmark (`alfa_test.csv`) is strictly held out and never accessed.

## 1. Inspect Hardware & Auto-Detect GPU Environment

In [ ]:
# Inspect GPU type and available VRAM
!nvidia-smi

import torch
if not torch.cuda.is_available():
    raise SystemError("CUDA GPU not detected! In Google Colab, go to Runtime > Change runtime type > Select T4, L4, or A100 GPU.")

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"\nDetected GPU: {gpu_name} ({total_vram_gb:.2f} GB VRAM)")

# Adaptive configuration based on detected VRAM
# Token length analysis confirmed max tokens in dataset = 445 tokens (Line 21625).
# Setting MAX_SEQ_LENGTH = 512 guarantees 100.0% coverage with zero command truncation.
if total_vram_gb >= 20.0:
    print("\n[Configuration] High-VRAM GPU (L4 / A100): Using 16-bit LoRA with batch size 8.")
    LOAD_IN_4BIT = False
    BATCH_SIZE = 8
    GRAD_ACCUM = 2
    MAX_SEQ_LENGTH = 512
    COMPUTE_DTYPE = torch.bfloat16
elif total_vram_gb >= 14.0:
    print("\n[Configuration] Standard T4 / V100 (15-16 GB): Using 4-bit QLoRA (NF4) with batch size 4.")
    LOAD_IN_4BIT = True
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
    MAX_SEQ_LENGTH = 512
    COMPUTE_DTYPE = torch.float16 if not torch.cuda.is_bf16_supported() else torch.bfloat16
else:
    print("\n[Configuration] Conservative Fallback (< 14 GB): Using 4-bit QLoRA with batch size 2.")
    LOAD_IN_4BIT = True
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    MAX_SEQ_LENGTH = 384
    COMPUTE_DTYPE = torch.float16

print(f"Active Settings: 4-bit={LOAD_IN_4BIT}, BatchSize={BATCH_SIZE}, GradAccum={GRAD_ACCUM}, MaxSeqLen={MAX_SEQ_LENGTH}, Dtype={COMPUTE_DTYPE}")

## 2. Install Pinned SFT & PEFT Dependencies

In [ ]:
# Install pinned mutually-compatible training libraries for Google Colab
!pip install -q "transformers>=4.46.0,<4.49.0" "peft>=0.13.0,<0.15.0" "trl>=0.11.0,<0.13.0" "bitsandbytes>=0.44.0" "accelerate>=0.34.0,<1.0.0" "datasets>=2.21.0,<3.0.0" pyyaml


## 3. Mount Google Drive for Persistent Storage & Resumption

Mounting Google Drive ensures that checkpoints and the final LoRA adapter survive Colab runtime disconnects.

In [ ]:
import os
from pathlib import Path

USE_GOOGLE_DRIVE = True  # Set to True to persist checkpoints and adapter directly to Google Drive

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_ROOT = Path('/content/drive/MyDrive/NL2SH_SFT')
        OUTPUT_DIR = str(DRIVE_ROOT / 'checkpoints')
        ADAPTER_DIR = str(DRIVE_ROOT / 'adapters' / 'qwen2.5-coder-3b-nl2sh-sft')
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        os.makedirs(ADAPTER_DIR, exist_ok=True)
        print(f"[Persistent Storage] Google Drive mounted successfully!")
        print(f"  Checkpoints directory: {OUTPUT_DIR}")
        print(f"  Adapter directory:     {ADAPTER_DIR}")
    except Exception as e:
        print(f"[Notice] Google Drive mount skipped or unavailable ({e}). Using local Colab storage.")
        OUTPUT_DIR = "checkpoints/qwen2.5-coder-3b-nl2sh-sft"
        ADAPTER_DIR = "adapters/qwen2.5-coder-3b-nl2sh-sft"
else:
    OUTPUT_DIR = "checkpoints/qwen2.5-coder-3b-nl2sh-sft"
    ADAPTER_DIR = "adapters/qwen2.5-coder-3b-nl2sh-sft"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

## 4. Clone NL2SH Repository & Verify Processed Datasets

In [ ]:
# Clone repository if running in a fresh Colab workspace
if not Path("data/processed/train.jsonl").exists():
    !git clone https://github.com/gurupawan265/NL2SH.git
    os.chdir("NL2SH")

train_path = Path("data/processed/train.jsonl")
val_path = Path("data/processed/val.jsonl")

assert train_path.exists(), f"Training data missing: {train_path}"
assert val_path.exists(), f"Validation data missing: {val_path}"

train_lines = [l for l in train_path.read_text(encoding='utf-8').splitlines() if l.strip()]
val_lines = [l for l in val_path.read_text(encoding='utf-8').splitlines() if l.strip()]

print(f"Verified Clean Training Records:   {len(train_lines):,}")
print(f"Verified Clean Validation Records: {len(val_lines):,}")
print(f"Total SFT Dataset:                {len(train_lines) + len(val_lines):,}")
print("Safety Check: Official test benchmark ('alfa_test.csv') remains isolated and untouched.")

## 5. Setup Chat Template & Format SFT Datasets

Each record is mapped into the Qwen2.5 ChatML format:
- **System:** Instructs the model to output exact Bash commands only.
- **User:** Natural-language instruction describing the task.
- **Assistant:** Ground-truth Bash command followed by `<|im_end|>`.

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are an expert Linux shell assistant. Your task is to translate natural language "
    "instructions into exact, executable Bash commands. Output only the command with no markdown, "
    "explanation, or preamble."
)

def format_qwen_chat(instruction: str, command: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{instruction.strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n{command.strip()}<|im_end|>"
    )

def load_dataset_from_jsonl(path):
    texts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                texts.append(format_qwen_chat(row['instruction'], row['command']))
    return Dataset.from_dict({"text": texts})

print("Formatting datasets into Qwen ChatML...")
train_dataset = load_dataset_from_jsonl("data/processed/train.jsonl")
val_dataset = load_dataset_from_jsonl("data/processed/val.jsonl")

print(f"Train dataset samples: {len(train_dataset):,}")
print(f"Val dataset samples:   {len(val_dataset):,}")
print("\n--- Sample ChatML Record ---")
print(train_dataset[0]["text"])

## 6. Load Base Model with QLoRA Configuration

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

# 1. Tokenizer Setup
print(f"Loading tokenizer: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Quantization Setup
bnb_config = None
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

# 3. Base Model Loading
print(f"Loading base model: {MODEL_ID}...")
model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": COMPUTE_DTYPE,
    "device_map": "auto",
}
if bnb_config is not None:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

if LOAD_IN_4BIT:
    model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()

# 4. PEFT LoRA Configuration (Target all 7 projection modules)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
print("\n--- Trainable Parameters ---")
model.print_trainable_parameters()

## 7. Configure Training Arguments & Loss Masking Collator

In [ ]:
from transformers import TrainingArguments
from trl import DataCollatorForCompletionOnlyLM, SFTTrainer

# Response-only loss masking: masks system and user tokens with -100,
# ensuring gradients are computed ONLY on the assistant's Bash command tokens.
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    fp16=(COMPUTE_DTYPE == torch.float16),
    optim="paged_adamw_8bit" if LOAD_IN_4BIT else "adamw_torch",
    seed=42,
    report_to="none",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=collator,
    tokenizer=tokenizer,
    args=training_args,
)

print("SFTTrainer with DataCollatorForCompletionOnlyLM initialized successfully.")

## 8. Execute Fine-Tuning with Checkpoint Resumption Support

In [ ]:
import glob
import time

# Check for existing checkpoints in case of interrupted training
existing_checkpoints = sorted(
    glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
    key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0
)
resume_checkpoint = existing_checkpoints[-1] if existing_checkpoints else None

if resume_checkpoint:
    print(f"[Checkpoint Resumption] Found existing checkpoint: {resume_checkpoint}")
    print(f"[Checkpoint Resumption] Resuming training from step {resume_checkpoint.split('-')[-1]}...")
else:
    print("[Training] No previous checkpoints found. Starting fresh SFT training...")

# Track execution time and peak memory
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
start_time = time.time()

# Run training
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
training_duration_sec = time.time() - start_time
training_duration_min = training_duration_sec / 60

# Record peak VRAM
peak_vram_allocated_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
peak_vram_reserved_gb = torch.cuda.max_memory_reserved() / (1024 ** 3) if torch.cuda.is_available() else 0.0

print(f"\n========================================")
print(f"TRAINING EXECUTION SUMMARY")
print(f"========================================")
print(f"GPU Model:             {gpu_name}")
print(f"Peak Allocated VRAM:   {peak_vram_allocated_gb:.2f} GB")
print(f"Peak Reserved VRAM:    {peak_vram_reserved_gb:.2f} GB")
print(f"Training Duration:     {training_duration_min:.2f} minutes ({training_duration_sec:.1f}s)")
print(f"Total Global Steps:    {trainer.state.global_step}")
print(f"========================================")

# Save final PEFT adapter and tokenizer to persistent directory
print(f"\nSaving fine-tuned adapter to {ADAPTER_DIR}...")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter and tokenizer successfully saved!")

## 9. Tabulate Training & Validation Loss History

In [ ]:
# Extract and display loss logs from trainer state
log_history = trainer.state.log_history

train_losses = [entry for entry in log_history if 'loss' in entry]
eval_losses = [entry for entry in log_history if 'eval_loss' in entry]

final_train_loss = train_losses[-1]['loss'] if train_losses else None
final_eval_loss = eval_losses[-1]['eval_loss'] if eval_losses else None

print(f"Final Training Loss:   {final_train_loss:.4f}" if final_train_loss else "Final Training Loss: N/A")
print(f"Final Validation Loss: {final_eval_loss:.4f}" if final_eval_loss else "Final Validation Loss: N/A")

print("\n--- Evaluation Loss Checkpoint Milestones ---")
print(f"{'Step':>8} | {'Epoch':>6} | {'Eval Loss':>10} | {'Eval Runtime (s)':>16}")
print("-" * 48)
for entry in eval_losses:
    step = entry.get('step', '-')
    epoch = f"{entry.get('epoch', 0.0):.2f}"
    loss = f"{entry.get('eval_loss', 0.0):.4f}"
    runtime = f"{entry.get('eval_runtime', 0.0):.2f}"
    print(f"{step:>8} | {epoch:>6} | {loss:>10} | {runtime:>16}")

## 10. Inference Verification on 10 Development Tasks

We evaluate the fine-tuned adapter on 10 diverse development instructions spanning common Linux domains:
file search, stream editing, process inspection, permission management, compression, network checks, and text aggregation.
(The official 300-record test set is held out strictly).

In [ ]:
from peft import PeftModel

# Function to generate raw bash commands without markdown filler
def generate_command(instruction: str, max_new_tokens: int = 64) -> str:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{instruction.strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.encode("<|im_end|>")[0],
        )
    raw_out = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return raw_out.strip()

# 10 Diverse Development Instructions
dev_benchmark_tasks = [
    ("File Search", "Find all '.log' files in /var/log modified within the last 7 days"),
    ("Pattern Search", "Search case-insensitively for 'FATAL ERROR' in /workspace/app.log"),
    ("Stream Replacement", "Replace all occurrences of 'localhost' with '127.0.0.1' in /workspace/config.env"),
    ("Permissions", "Recursively set permissions of all directories in /workspace to 755"),
    ("Archiving & Compression", "Create a gzip-compressed tar archive named backup.tar.gz from /workspace/data"),
    ("Process Management", "Find all processes named 'nginx' and gracefully reload them"),
    ("Disk Usage Inspection", "Display human-readable disk usage of each subfolder in /workspace sorted by size"),
    ("Line Counting", "Count the number of non-empty lines in /workspace/sample.txt"),
    ("Deduplication & Frequency", "Extract unique IP addresses from /workspace/access.log and count occurrences sorted descending"),
    ("Networking / Health Check", "Send an HTTP GET request to http://localhost:8080/health with a 5 second timeout"),
]

print("================================================================================")
print("POST-TRAINING INFERENCE VERIFICATION (10 DEVELOPMENT INSTRUCTIONS)")
print("================================================================================\n")

for idx, (category, instruction) in enumerate(dev_benchmark_tasks, start=1):
    generated_cmd = generate_command(instruction)
    print(f"[{idx:02d}] Category:    {category}")
    print(f"     Instruction: {instruction}")
    print(f"     Generated:   {generated_cmd}\n")

## 11. Export Adapter Archive & Confirm Persistence

In [ ]:
# Zip adapter directory for easy local download or archive
!zip -r qwen2.5_coder_3b_nl2sh_adapter.zip {ADAPTER_DIR}

print(f"\n[Status] Adapter archived to: qwen2.5_coder_3b_nl2sh_adapter.zip")
print(f"[Status] Direct persistent location: {ADAPTER_DIR}")

try:
    from google.colab import files
    files.download('qwen2.5_coder_3b_nl2sh_adapter.zip')
    print("[Status] Download initiated in Colab browser.")
except Exception as e:
    print(f"[Notice] Colab browser download not triggered ({e}).")